# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Este cuaderno audita el sistema de etiquetas reales (`post_tag`) de Revista Anfibia extraídas de la base de datos original.
Comparamos las etiquetas manuales frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento, y generar una propuesta de auto-etiquetado.

In [7]:
# 1. Parche de compatibilidad obligatorio
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
import warnings
warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
RAW_DATA = PROJECT_ROOT / "data" / "processed" / "posteos_anfibia.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos
df_corpus = pd.read_csv(DATA_PROCESSED)
df_tags = pd.read_csv(RAW_DATA, usecols=["ID", "post_tag"])

# Merge para traer los tags reales
df = df_corpus.merge(df_tags, on="ID", how="left")
df["doc_id"] = df.index

# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topics, _ = topic_model.transform(df["document_text_final"].fillna("").tolist())
df["topic"] = topics

topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()
df["topic_name"] = df["topic"].map(
    lambda t: ", ".join(topic_info.get(t, "Outlier").split("_")[1:4]) if t != -1 else "Outlier / Ruido"
)

print(f"Total de notas cruzadas: {len(df):,}")

Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [8]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cobertura de Etiquetas en el Archivo", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [13]:
import plotly.graph_objects as go

# 1. Traemos el año de publicación directo del archivo original
df_fechas = pd.read_csv(RAW_DATA, usecols=["ID", "post_year"])
df_evolucion = df.merge(df_fechas, on="ID", how="left")

# 2. Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df_evolucion.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos (ej. 1970)
evolucion = evolucion[evolucion["post_year"] > 2010]

# 3. Gráfico de barras apiladas
fig_evo = go.Figure()

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=evolucion["pct_con_tag"].astype(str) + "%",
    textposition='inside'
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas (Vacío)",
    marker_color='#d62728'
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Degradación del Etiquetado: Notas con y sin Tags por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category') # Para que los años se vean discretos
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [9]:
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de tags únicos creados: {tags_unicos:,}")
print(f"Tags usados en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print("\nTop 15 Tags más usados:")
display(tag_counts.head(15).to_frame("Frecuencia"))

Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)

Top 15 Tags más usados:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [10]:
# Armamos df desanidado completo para agrupar
df_desanidado = df.drop("post_tag", axis=1).join(
    df["post_tag"].dropna().str.split(r";").explode().str.strip().rename("tag")
).dropna(subset=["tag"])

df_desanidado = df_desanidado[df_desanidado["tag"] != ""]

# Agrupamos por tópico
solapamiento = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,"político, política, discurso",544,416,Perfil político | Memoria trans | 24M: Día de ...
4,4,"papal, catolicismo, arzobispo",33,31,Papa Francisco | Rossana Reguillo | Catolicism...
1,1,"noticia, sociedad, tecnológico",32,52,Rossana Reguillo | Grupo Clarín | Medios de co...
3,3,"pandemia, epidemia, vacunar",24,16,Rosariocronica | Rosario | Covid-19 | Escuela ...
2,2,"aborto, abortar, anticonceptivo",22,28,Aborto Legal | LGBTIQ+ | Memoria trans | Orgul...
7,7,"israel, netanyahu, israelí",18,24,Internacional | Israel | Palestina | 24M: Día ...
6,6,"soviética, soviético, vladimir",16,21,Internacional | Rusia | Barras bravas | Fútbol...
5,5,"arte, art, estética",7,14,Moda y medios | Covid-19 | Escuela de Arte y P...
9,9,"submarino, marino, océano",5,4,Malvinas | Submarinos | ARA San Juan | Océanos
10,10,"festejo, manifestación, fiesta",3,11,2001 | A 20 años del 2001 | Fotoperiodismo | P...


## 4. Normalización: Generación de Etiquetas
El modelo resuelve el vacío del 56% de notas sin tag y unifica la dispersión semántica asignando etiquetas normalizadas.

In [ ]:
# Usamos directamente las 3 palabras clave más representativas que calculó BERTopic 
# (esto ya está en la columna 'topic_name' que creamos)
df["tag_canonico_propuesto"] = df["topic_name"]

# Para los outliers (ruido estadístico), los dejamos explícitamente sin clasificar
df.loc[df["topic"] == -1, "tag_canonico_propuesto"] = np.nan

# Muestra comparativa: Tag manual de la redacción vs Tag algorítmico
display(df[df["tiene_tag"]][["title_text", "post_tag", "tag_canonico_propuesto"]].head(15))

# Exportar el dataset enriquecido
export_cols = ["doc_id", "ID", "title_text", "post_tag", "topic", "tag_canonico_propuesto"]
df[export_cols].to_csv(OUTPUT_DIR / "corpus_con_tags_normalizados.csv", index=False)
print("Archivo 'corpus_con_tags_normalizados.csv' exportado.")

,title_text,post_tag,tag_canonico_propuesto
1,El rico que se cree Steve Jobs,Perfil político,"político, política, discurso"
17,Fluidos trans: arte y performance queer,Memoria trans,"político, política, discurso"
28,¿Qué tan lejos llegan en 15 segundos?,Internacional; Israel; Palestina,"israel, netanyahu, israelí"
33,El sol de tu bravura,Internacional; Venezuela,Sin clasificar / Ruido
40,La 12: historia de una traición,Barras bravas; Boca Juniors; Fútbol,Sin clasificar / Ruido
45,Policías en acción,Violencia policial,Sin clasificar / Ruido
64,Bolivia: Los linchados de El Alto,BA,Sin clasificar / Ruido
83,Enfrentar la muerte,24M: Día de la memoria por la verdad y la just...,"político, política, discurso"
90,La narcomáquina ya no necesita Chapos,Rossana Reguillo,Sin clasificar / Ruido
92,"Crimea: allí donde haya rusos, hay Rusia",Internacional; Rusia,"soviética, soviético, vladimir"


Archivo 'corpus_con_tags_normalizados.csv' exportado.
